# Week 2: NomNaOCR EDA and character-level edit distance

This notebook has two evidence levels: (1) an inventory of the read-only full NomNaOCR tree and (2) detailed inspection of the five page pairs prepared in Week 1. It does not write into the shared dataset. Run every cell, record observations in the response cells, and retain the outputs for submission.

## 1. From the two papers to one problem formulation

Use the NomNaOCR paper to identify what data and supervision exist. Use the automated-annotation paper to identify what character-level artifact is missing and how detection, radical-based recognition, validation, and iterative updating interact. Before coding, write a team answer:

- **Input and unit of analysis:** …
- **Available supervision:** …
- **Desired output:** …
- **Success measures:** …
- **Key assumptions:** …
- **Largest failure risk:** …

Your formulation should distinguish a page, a labeled text line/column, a detected character box, a Unicode character, and an IDS token sequence.

In [ ]:
import os
import unicodedata
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from PIL import Image
from nom_ocr_lab import discover_page_pairs, read_line_annotations
from nom_ocr_lab.annotations import parse_line_annotation

if "LAB_WORKSPACE" not in os.environ:
    raise RuntimeError("Set LAB_WORKSPACE=/data/projectgX/lab-exercise before starting Jupyter")

FULL_ROOT = Path(os.environ.get("NOMNAOCR_ROOT", "/data/shared/project2026/nomnaocr"))
SAMPLE_ROOT = Path(os.environ["LAB_WORKSPACE"]) / "datasets" / "nomnaocr-sample"

if not FULL_ROOT.is_dir():
    raise FileNotFoundError(f"Full dataset not found or unreadable: {FULL_ROOT}")
if not (SAMPLE_ROOT / "manifest.json").is_file():
    raise FileNotFoundError(
        f"Week 1 sample not found: {SAMPLE_ROOT}\n"
        "Prepare it with labs/week-01-orientation/prepare_sample.sbatch; "
        "the exact commands are in this week's README."
    )

page_pairs = discover_page_pairs(SAMPLE_ROOT)
assert len(page_pairs) == 5, f"Expected 5 sample page pairs, found {len(page_pairs)}"
print(f"Full dataset (read only): {FULL_ROOT}")
print(f"Week 1 sample: {SAMPLE_ROOT}")
print(f"Matched sample pairs: {len(page_pairs)}")

## 2. Filesystem and pairing rule

The course copy is expected to have this logical layout (the code below reports the actual tree):

```text
nomnaocr/
├── images/<document>/<page>.<image extension>
├── line_labels/<document>/<page>.txt
└── detection_labels/<document>/<page>.txt
```

An image and label pair share the same path relative to their respective roots after replacing the extension with `.txt`. Directory names preserve the source document/volume. `line_labels` contain quadrilateral line regions plus transcription. Inspect `detection_labels` empirically before claiming its schema; a common five-field row is YOLO's `class x_center y_center width height`, but the values and documentation must confirm that interpretation.

In [ ]:
def human_bytes(number):
    value = float(number)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if value < 1024 or unit == "TiB":
            return f"{value:.2f} {unit}"
        value /= 1024

def files_below(root):
    return sorted(path for path in root.rglob("*") if path.is_file()) if root.is_dir() else []

print("Actual top-level entries:")
for path in sorted(FULL_ROOT.iterdir()):
    print(f"  {path.name}/" if path.is_dir() else f"  {path.name}")

inventory = {}
for directory_name in ("images", "line_labels", "detection_labels"):
    paths = files_below(FULL_ROOT / directory_name)
    inventory[directory_name] = paths
    suffixes = Counter(path.suffix.lower() or "<none>" for path in paths)
    total_bytes = sum(path.stat().st_size for path in paths)
    print(f"{directory_name:>16}: {len(paths):>6} files, {human_bytes(total_bytes):>10}, formats={dict(suffixes)}")

full_pairs = discover_page_pairs(FULL_ROOT)
print(f"Matched image + line-label pairs: {len(full_pairs)}")
print("First five relative image paths:")
for pair in full_pairs[:5]:
    print(f"  {pair.relative_image}")

### Inventory observations

Record the observed counts and sizes. The NomNaOCR paper reports **2,953 pages** and **38,318 text patches**. Explain any mismatch rather than forcing the filesystem to match the publication: the course copy may represent pages, lines, character detections, or derived labels differently.

- Observed page/image count and total bytes: …
- Observed line-label count and total rows: …
- Observed detection-label count and interpretation: …
- Missing/unmatched files or count discrepancy: …

In [ ]:
# Reading dimensions requires decoding images, so inspect a documented prefix rather than all pages.
image_probe = [pair.image_path for pair in full_pairs[:200]]
image_modes = Counter()
image_dimensions = Counter()
for image_path in image_probe:
    with Image.open(image_path) as image:
        image_modes[image.mode] += 1
        image_dimensions[image.size] += 1
print(f"Decoded image metadata for {len(image_probe)} of {len(inventory['images'])} files")
print(f"Image modes: {dict(image_modes)}")
print(f"Most common dimensions: {image_dimensions.most_common(10)}")

line_files = inventory["line_labels"]
line_rows = malformed_rows = question_mark_rows = transcribed_characters = 0
malformed_examples = []
for label_path in line_files:
    for line_number, raw in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        if not raw.strip():
            continue
        line_rows += 1
        try:
            annotation = parse_line_annotation(raw)
        except ValueError as error:
            malformed_rows += 1
            if len(malformed_examples) < 5:
                malformed_examples.append((str(label_path), line_number, str(error)))
            continue
        question_mark_rows += "?" in annotation.text
        transcribed_characters += len(annotation.text)
print(f"Non-empty line-label rows: {line_rows}")
print(f"Transcribed Unicode code points: {transcribed_characters}")
print(f"Rows containing '?': {question_mark_rows}")
print(f"Malformed rows: {malformed_rows}; examples={malformed_examples}")

detection_files = inventory["detection_labels"]
if detection_files:
    raw_detection_rows = [row for row in detection_files[0].read_text(encoding="utf-8").splitlines() if row.strip()]
    print(f"\nFirst detection-label file: {detection_files[0]}")
    for row in raw_detection_rows[:5]:
        print(f"  fields={len(row.split())}: {row}")
else:
    print("\nNo detection_labels directory/files found; record this as an observed dataset difference.")

## 3. Label conventions and five-page visual audit

Each non-empty line-label row is parsed as `x1,y1,x2,y2,x3,y3,x4,y4,text`. The first eight fields are four pixel-coordinate vertices of a quadrilateral; the ninth is the Unicode transcription. Splitting is limited to eight commas so punctuation in the transcription is preserved. A `?` is evidence of missing, illegible, or uncertain digital transcription—not an ordinary character class. File order and polygon geometry must be checked against the actual vertical reading order.

For each page below, inspect: blur/fading, ink bleed or stains, tears/cropping/skew, vertical columns versus mixed layout, marginal notes or headers, polygon tightness/overlap, reading order, `?`, punctuation, and unusual/variant characters. Do not infer quality from filename alone.

In [ ]:
sample_annotations = {}
for page_number, pair in enumerate(page_pairs, start=1):
    annotations = read_line_annotations(pair.label_path)
    sample_annotations[pair.relative_image.as_posix()] = annotations
    with Image.open(pair.image_path) as source_image:
        image = source_image.convert("RGB")

    print(f"\nPage {page_number}: {pair.relative_image}")
    print(f"format={pair.image_path.suffix.lower()}, pixels={image.width}x{image.height}, lines={len(annotations)}")
    for line_number, annotation in enumerate(annotations, start=1):
        print(f"  {line_number:>2}: {annotation.text!r} | {annotation.points}")

    fig, ax = plt.subplots(figsize=(8, 11))
    ax.imshow(image)
    for line_number, annotation in enumerate(annotations, start=1):
        ax.add_patch(Polygon(annotation.points, closed=True, fill=False, edgecolor="red", linewidth=1.2))
        ax.text(annotation.points[0][0], annotation.points[0][1], str(line_number), color="yellow", fontsize=8, bbox={"facecolor": "black", "alpha": 0.6, "pad": 1})
    ax.set_title(f"Page {page_number}: {pair.relative_image} ({len(annotations)} lines)")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

### Visual observations (complete after viewing)

| Page | Quality/degradation evidence | Layout and likely reading order | Label/convention issue | Pipeline risk |
| --- | --- | --- | --- | --- |
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |

Which observation should become a project risk, who owns it, and what measurable trigger will activate a contingency? …

## 4. Character-level distances and similarities

All functions below operate on Python strings, so the sequence elements are Unicode code points. NFC normalization is applied before page comparison; it does not resolve historical glyph variants or semantically equivalent code points.

**Hamming distance** (equal lengths only): $d_H(x,y)=\sum_i [x_i \ne y_i]$. It counts substitutions but cannot represent insertion or deletion.

**Levenshtein distance** uses insertion, deletion, and substitution:

$$D_{i,j}=\min\{D_{i-1,j}+1,\;D_{i,j-1}+1,\;D_{i-1,j-1}+[x_i\ne y_j]\}.$$

**Optimal-string-alignment (restricted Damerau–Levenshtein)** also permits one adjacent transposition, adding $D_{i-2,j-2}+1$ when the last two symbols are swapped. Unlike unrestricted Damerau–Levenshtein, a substring cannot be edited more than once.

**LCS distance** derives from the longest common subsequence: $d_{LCS}(x,y)=|x|+|y|-2|LCS(x,y)|$. It models insertions and deletions, not substitution as a single edit.

For Levenshtein/OSA we use $s=1-d/\max(|x|,|y|)$. For LCS we use $s=2|LCS|/(|x|+|y|)$. Both return 1 for two empty sequences. Always report the normalization with a similarity score.

In [ ]:
def hamming_distance(left, right):
    if len(left) != len(right):
        raise ValueError("Hamming distance requires equal-length sequences")
    return sum(a != b for a, b in zip(left, right))

def levenshtein_distance(left, right):
    # Two rows are enough when only the final distance is required.
    previous = list(range(len(right) + 1))
    for i, left_item in enumerate(left, start=1):
        current = [i]
        for j, right_item in enumerate(right, start=1):
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + (left_item != right_item)))
        previous = current
    return previous[-1]

def osa_damerau_levenshtein_distance(left, right):
    table = [[0] * (len(right) + 1) for _ in range(len(left) + 1)]
    for i in range(len(left) + 1):
        table[i][0] = i
    for j in range(len(right) + 1):
        table[0][j] = j
    for i in range(1, len(left) + 1):
        for j in range(1, len(right) + 1):
            table[i][j] = min(table[i - 1][j] + 1, table[i][j - 1] + 1, table[i - 1][j - 1] + (left[i - 1] != right[j - 1]))
            if i > 1 and j > 1 and left[i - 1] == right[j - 2] and left[i - 2] == right[j - 1]:
                table[i][j] = min(table[i][j], table[i - 2][j - 2] + 1)
    return table[-1][-1]

def lcs_length(left, right):
    previous = [0] * (len(right) + 1)
    for left_item in left:
        current = [0]
        for j, right_item in enumerate(right, start=1):
            current.append(previous[j - 1] + 1 if left_item == right_item else max(previous[j], current[j - 1]))
        previous = current
    return previous[-1]

def lcs_distance(left, right):
    return len(left) + len(right) - 2 * lcs_length(left, right)

def normalized_edit_similarity(left, right, distance=levenshtein_distance):
    denominator = max(len(left), len(right))
    return 1.0 if denominator == 0 else 1.0 - distance(left, right) / denominator

def lcs_similarity(left, right):
    denominator = len(left) + len(right)
    return 1.0 if denominator == 0 else 2.0 * lcs_length(left, right) / denominator

assert hamming_distance("Nôm", "Nơm") == 1
assert levenshtein_distance("kitten", "sitting") == 3
assert osa_damerau_levenshtein_distance("ab", "ba") == 1
assert lcs_length("ABCBDAB", "BDCABA") == 4
assert normalized_edit_similarity("", "") == 1.0
print("All distance sanity checks passed.")

In [ ]:
# Preserve line boundaries and normalize canonically; do not discard punctuation or '?'.
page_names = [pair.relative_image.as_posix() for pair in page_pairs]
page_texts = {
    name: unicodedata.normalize("NFC", "\n".join(item.text for item in sample_annotations[name]))
    for name in page_names
}
for index, name in enumerate(page_names, start=1):
    print(f"Page {index}: {len(page_texts[name])} code points (including line separators), {name}")

def pairwise_similarity(similarity_function):
    return [[similarity_function(page_texts[left], page_texts[right]) for right in page_names] for left in page_names]

similarity_matrices = {
    "Normalized Levenshtein": pairwise_similarity(normalized_edit_similarity),
    "Normalized OSA Damerau–Levenshtein": pairwise_similarity(lambda a, b: normalized_edit_similarity(a, b, osa_damerau_levenshtein_distance)),
    "LCS similarity": pairwise_similarity(lcs_similarity),
}

for title, matrix in similarity_matrices.items():
    print(f"\n{title}")
    print("       " + " ".join(f"P{i:>5}" for i in range(1, 6)))
    for i, row in enumerate(matrix, start=1):
        print(f"P{i:<5} " + " ".join(f"{value:6.3f}" for value in row))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (title, matrix) in zip(axes, similarity_matrices.items()):
    image = ax.imshow(matrix, vmin=0, vmax=1, cmap="viridis")
    ax.set_title(title)
    ax.set_xticks(range(5), [f"P{i}" for i in range(1, 6)])
    ax.set_yticks(range(5), [f"P{i}" for i in range(1, 6)])
    for row in range(5):
        for column in range(5):
            ax.text(column, row, f"{matrix[row][column]:.2f}", ha="center", va="center", color="white" if matrix[row][column] < 0.55 else "black", fontsize=8)
fig.colorbar(image, ax=axes, shrink=0.8, label="similarity")
plt.show()

## 5. Interpret, then transfer findings to the charter

- Which off-diagonal pair is most similar under each formula? Is the ranking stable? …
- Why can Levenshtein and LCS disagree? Give one concrete edit pattern. …
- How do page length, line separators, punctuation, `?`, Unicode normalization, and variant code points affect the scores? …
- Why are these transcription similarities **not** visual similarity, OCR accuracy, or proof that two pages have the same layout? …
- Later, when predictions are compared with ground truth, which distance/normalization would you choose and why? …

Finally, move evidence—not vague concerns—into `project-charter-template.md`: at least three testable goals, explicit non-goals, an owner and backup/reviewer for each workstream, and risks with mitigation plus observable triggers.